# 05 Robustness

Robustness check with a second buyer: PEPFAR/SCMS HIV shipments 2006-2015 (USAID).

Compares SCMS with the Global Fund PQR restricted to the same scope (HIV/ARV medicines,
purchase/delivery years 2006-2015):
  1. supplier concentration and overlap of leading firms;
  2. product relatedness: correlation of Hidalgo proximity (phi) on common product pairs,
     with a Mantel-type permutation test;
  3. community structure: Louvain modularity vs 200 degree-preserving rewirings;
  4. entry: does density predict a firm's entry into new ARV products (cutoffs 2009, 2012)?
Firm = manufacturing site mapped with the same firm rules; product = molecule combination
(normalized so SCMS "Lamivudine/Zidovudine" == PQR "Lamivudine+Zidovudine - FDC").


f10: product proximity in Global Fund PQR vs PEPFAR SCMS (HIV ARVs, 2006-2015).

In [ ]:
%run 00_utils.ipynb

import re
import numpy as np, pandas as pd, statsmodels.formula.api as smf, matplotlib.pyplot as plt

set_style()

## Comparison

In [ ]:
rng = np.random.default_rng(5205)
MIN_TX = 2

MOLECULE_ALIASES = {"tenofovir disoproxil fumarate": "tenofovir", "tenofovir df": "tenofovir", "tdf": "tenofovir",
         "didanosine ec": "didanosine", "nelfinavir as mesilate": "nelfinavir"}
def prod_key(name):
    name = name.lower().replace(" - fdc", " ").replace(" fdc", " ").replace("co-blister", " ")
    name = re.sub(r"\(.*?\)", " ", name)
    molecules = [re.sub(r"[^a-z ]", " ", part).strip() for part in re.split(r"[+/\[\]]", name)]
    molecules = [MOLECULE_ALIASES.get(re.sub(r"\s+", " ", part), re.sub(r"\s+", " ", part)) for part in molecules if part.strip()]
    return " + ".join(sorted(set(molecules)))

def firm_of(raw_name):
    matched = match(raw_name)
    return matched[0] if matched and matched[0] else re.split(r"[,(]", raw_name)[0].strip()

In [ ]:
# ---- SCMS
scms = pd.read_csv(ROOT / "data" / "raw" / "scms_delivery_history_2006-2015.csv")
scms = scms[scms["product group"] == "ARV"].copy()
scms["firm"] = scms["manufacturing site"].map(firm_of)
scms["product"] = scms["molecule/test type"].map(prod_key)
scms["year"] = pd.to_datetime(scms["delivered to client date"], format="%d-%b-%y", errors="coerce").dt.year
scms["value"] = pd.to_numeric(scms["line item value"], errors="coerce")
scms = scms.rename(columns={"id": "pk"})

I used the USAID SCMS Delivery History, which has 10,324 PEPFAR-funded shipment lines from 2006-2015, including 8,550 ARV lines. The official source is now offline, so I took the file from the Internet Archive copy of the official CSV. I compared it with PQR HIV transactions in the same years and defined products as molecule combinations in both sources.

In [ ]:
# ---- PQR, same scope
pqr = load_clean("pqr_medicines")
pqr = pqr[(pqr.area == "HIV/ARV") & pqr.year.between(2006, 2015)].copy()
pqr["product"] = pqr["product"].map(prod_key)
pqr["value"] = pqr.cost_usd

def firm_product_links(table, min_tx=MIN_TX):
    links = table.groupby(["firm", "product"]).agg(n_tx=("pk", "size"), first_year=("year", "min")).reset_index()
    return links[links.n_tx >= min_tx]

rows = []
def add(section, metric, pqr_val, scms_val, note=""):
    rows.append({"section": section, "metric": metric, "PQR (Global Fund)": pqr_val, "SCMS (PEPFAR)": scms_val, "note": note})

In [ ]:
# 1. scope and concentration
firm_share = lambda table: table.groupby("firm").value.sum() / table.value.sum()
share_pqr, share_scms = firm_share(pqr), firm_share(scms)
add("scope", "transactions (ARV, 2006-2015)", len(pqr), len(scms))
add("scope", "firms", pqr.firm.nunique(), scms.firm.nunique())
add("scope", "products (molecule combinations)", pqr["product"].nunique(), scms["product"].nunique())
add("scope", "value, USD m", round(pqr.value.sum() / 1e6), round(scms.value.sum() / 1e6))
add("concentration", "top-5 firm share of value", round(share_pqr.nlargest(5).sum(), 3), round(share_scms.nlargest(5).sum(), 3))
add("concentration", "top-10 firm share of value", round(share_pqr.nlargest(10).sum(), 3), round(share_scms.nlargest(10).sum(), 3))
top5_pqr, top5_scms = set(share_pqr.nlargest(5).index), set(share_scms.nlargest(5).index)
add("concentration", "top-5 firms", "; ".join(share_pqr.nlargest(5).index), "; ".join(share_scms.nlargest(5).index),
    f"{len(top5_pqr & top5_scms)} of 5 in common")
common_firms = share_pqr.index.intersection(share_scms.index)
add("concentration", "Spearman corr. of firm shares (common firms)",
    round(share_pqr[common_firms].corr(share_scms[common_firms], method="spearman"), 2), "", f"{len(common_firms)} common firms")
shared_products = set(pqr["product"]) & set(scms["product"])
add("scope", "products in both sources", len(shared_products), len(shared_products))

The two buyers saw a similar supplier market. The top 5 firms held 85% of value in PQR and 91% in SCMS, and 4 of the top 5 firms were the same. Firm shares across the 18 common firms had a Spearman correlation of 0.77.

In [ ]:
# 2. product relatedness
links_pqr, links_scms = firm_product_links(pqr), firm_product_links(scms)
def phi_matrix(links, common_products):
    proximity_edges = project_products(links)
    matrix = pd.DataFrame(0.0, index=common_products, columns=common_products)
    for product_a, product_b, proximity in proximity_edges[["source", "target", "phi"]].itertuples(index=False):
        if product_a in matrix.index and product_b in matrix.index: matrix.loc[product_a, product_b] = matrix.loc[product_b, product_a] = proximity
    return matrix
common_products = sorted(set(links_pqr["product"]) & set(links_scms["product"]))
proximity_pqr, proximity_scms = phi_matrix(links_pqr, common_products), phi_matrix(links_scms, common_products)
upper_pairs = np.triu_indices(len(common_products), 1)
pairs_pqr, pairs_scms = proximity_pqr.values[upper_pairs], proximity_scms.values[upper_pairs]
observed_corr = np.corrcoef(pairs_pqr, pairs_scms)[0, 1]
# mantel test: shuffle product labels to see how large a correlation chance can produce
permuted_corrs = []
for _ in range(2000):
    order = rng.permutation(len(common_products))
    permuted_corrs.append(np.corrcoef(pairs_pqr, proximity_scms.values[np.ix_(order, order)][upper_pairs])[0, 1])
p_mantel = (np.sum(np.array(permuted_corrs) >= observed_corr) + 1) / (len(permuted_corrs) + 1)
add("relatedness", "Pearson corr. of product proximity (common pairs)", round(observed_corr, 2), "",
    f"{len(common_products)} common products, {len(pairs_pqr)} pairs; Mantel permutation p = {p_mantel:.4f}")
add("relatedness", "Spearman corr. of product proximity", round(pd.Series(pairs_pqr).corr(pd.Series(pairs_scms), method="spearman"), 2), "")
pd.DataFrame({"pair": [f"{common_products[i]} | {common_products[j]}" for i, j in zip(*upper_pairs)], "phi_pqr": pairs_pqr, "phi_scms": pairs_scms}).to_csv(
    NETWORK / "scms_vs_pqr_phi.csv", index=False)

Product relatedness also looked similar. Proximity across 406 common product pairs correlated at r = 0.78 (Mantel p < 0.001). This suggests relatedness reflects what firms can make, not only Global Fund procurement rules.

In [ ]:
# 3. community structure vs null
for name, links in [("PQR (Global Fund)", links_pqr), ("SCMS (PEPFAR)", links_scms)]:
    network = product_graph(project_products(links), links["product"].unique())
    modularity_q = best_louvain(network, range(30))[0][1]
    null_q = []
    for _ in range(200):
        null_links = rewire_links(links, 10 * len(links), rng)
        null_network = product_graph(project_products(null_links), null_links["product"].unique())
        null_q.append(best_louvain(null_network, rng.integers(1e6, size=3))[0][1])
    z_score = (modularity_q - np.mean(null_q)) / np.std(null_q)
    rows.append({"section": "communities", "metric": f"modularity Q / null mean / z  [{name}]",
                 "PQR (Global Fund)" if name.startswith("PQR") else "SCMS (PEPFAR)": f"{modularity_q:.3f} / {np.mean(null_q):.3f} / {z_score:.1f}",
                 "note": f"p = {(np.sum(np.array(null_q) >= modularity_q) + 1) / 201:.3f}; {network.number_of_nodes()} products, {links.firm.nunique()} firms"})

Within HIV alone, neither network had communities beyond chance (PQR Q = 0.12 vs null 0.13; SCMS Q = 0.07 vs null 0.11). This fits the main analysis, where HIV products form one dense core. The communities only appear when all three diseases are pooled.

In [ ]:
# 4. entry vs density (SCMS only)
# same design as 04_entry, using SCMS firms and products
def entry_panel_at(scms_links, cutoff_year, H=3, recent_df=None):
    state = scms_links[scms_links.first_year < cutoff_year]
    proximity_edges = project_products(state)
    phi = pd.concat([proximity_edges.rename(columns={"source": "p", "target": "q"}), proximity_edges.rename(columns={"target": "p", "source": "q"})])
    total_proximity = phi.groupby("p").phi.sum()
    existing_links = set(zip(state.firm, state["product"]))
    recent = recent_df[recent_df.year.between(cutoff_year - 3, cutoff_year - 1)]
    pairs = pd.MultiIndex.from_product([recent.firm.unique(), recent["product"].unique()], names=["firm", "product"]).to_frame(index=False)
    pairs = pairs[[(f, q) not in existing_links for f, q in zip(pairs.firm, pairs["product"])]]
    covered = state[["firm", "product"]].rename(columns={"product": "q"}).merge(phi, on="q").groupby(["firm", "p"]).phi.sum().rename("cov").reset_index()
    pairs = pairs.merge(covered, left_on=["firm", "product"], right_on=["firm", "p"], how="left").drop(columns="p")
    pairs["density"] = (pairs["cov"].fillna(0) / pairs["product"].map(total_proximity)).fillna(0)
    pairs["firm_n"] = pairs.firm.map(state.groupby("firm")["product"].nunique()).fillna(0)
    pairs["prod_n"] = pairs["product"].map(state.groupby("product").firm.nunique()).fillna(0)
    future_links = set(zip(*scms_links[(scms_links.first_year >= cutoff_year) & (scms_links.first_year < cutoff_year + H)][["firm", "product"]].T.values))
    pairs["entry"] = [int((f, q) in future_links) for f, q in zip(pairs.firm, pairs["product"])]
    pairs["cutoff"] = cutoff_year
    return pairs
scms_links = scms.groupby(["firm", "product"]).agg(first_year=("year", "min")).reset_index()
# keep pairs with a known firm and product, then fit the logit with controls
entry_panel = pd.concat([entry_panel_at(scms_links, cutoff_year, recent_df=scms) for cutoff_year in [2009, 2012]])
entry_panel = entry_panel[(entry_panel.firm_n > 0) & (entry_panel.prod_n > 0)]
entry_panel["density_z"] = (entry_panel.density - entry_panel.density.mean()) / entry_panel.density.std()
entry_panel["log_firm_n"], entry_panel["log_prod_n"] = np.log(entry_panel.firm_n), np.log(entry_panel.prod_n)
entry_logit = smf.logit("entry ~ density_z + log_firm_n + log_prod_n + C(cutoff)", data=entry_panel).fit(disp=0)
density_ci = np.exp(entry_logit.conf_int().loc["density_z"])
logit_table = pd.read_csv(TABLES / "t10_entry_logit.csv")
pqr_density = logit_table[(logit_table.model == "M4 full") & (logit_table.term == "density_z")].iloc[0]
add("entry", "rows / entries (cutoffs 2009, 2012)", "", f"{len(entry_panel)} / {entry_panel.entry.sum()}")
add("entry", "median density: entrants vs non-entrants", "",
    f"{entry_panel[entry_panel.entry == 1].density.median():.2f} vs {entry_panel[entry_panel.entry == 0].density.median():.2f}")
add("entry", "OR per SD of density (logit, controls)", f"{pqr_density.odds_ratio:.2f} [{pqr_density.ci_low:.2f}, {pqr_density.ci_high:.2f}] (all diseases, t10 M4)",
    f"{np.exp(entry_logit.params['density_z']):.2f} [{density_ci[0]:.2f}, {density_ci[1]:.2f}], p = {entry_logit.pvalues['density_z']:.3f}")

out = pd.DataFrame(rows)
out.to_csv(TABLES / "t13_robustness_scms.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80)
print(out.fillna("").to_string(index=False))
print("\nSCMS unmatched firm names:", sorted(set(scms.firm) - set(pd.read_csv(PROCESSED / "firm_crosswalk.csv").firm.dropna())))

Entry followed density in PEPFAR purchases too (median density 0.39 for entrants vs 0.05 for non-entrants). One SD of density had an OR of 1.47 (95% CI 1.04-2.09), close to the 1.37 I found in PQR. There were only 39 entries, and the PQR number covers all diseases, so the comparison is rough. This check also only covers HIV up to 2015, with 18 firms and 29 products.

## Figure

In [ ]:
proximity_pairs = pd.read_csv(NETWORK / "scms_vs_pqr_phi.csv")
correlation = proximity_pairs.phi_pqr.corr(proximity_pairs.phi_scms)
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls="--", zorder=1)
ax.scatter(proximity_pairs.phi_pqr, proximity_pairs.phi_scms, s=26, color=BLUE, alpha=0.55, edgecolors="white", linewidths=0.6, zorder=2)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Proximity in Global Fund purchases (PQR)")
ax.set_ylabel("Proximity in PEPFAR purchases (SCMS)")
ax.text(0.03, 0.95, f"r = {correlation:.2f}\n{len(proximity_pairs)} product pairs\nMantel p {'< 0.001' if p_mantel < 0.001 else f'= {p_mantel:.3f}'}", transform=ax.transAxes, va="top", fontsize=9, color=INK)
add_title(ax, "Product relatedness looks the same to a different buyer",
          "Each dot = a pair of ARV products; proximity = how often the same firms supply both", size=11.5, sub_size=8.5)
fig.text(0.01, -0.03, f"HIV ARVs, 2006-2015, {len(common_products)} molecule combinations bought by both. Dashed line = equal proximity.\n"
         "Sources: Global Fund PQR; USAID SCMS Delivery History (archived copy).", fontsize=7.5, color=MUTED)
save_figure(fig, "f10_robustness_scms.png")